# Chuvas e Deslizamentos no Estado do Rio de Janeiro
## 1. Introdução
Chuvas intensas em áreas de encosta causam deslizamentos, desalojados e óbitos. Este notebook investiga **quais municípios são mais vulneráveis, quando o risco é maior e quanto a chuva explica os deslizamentos**.

## 2. Base de dados
Arquivo `dados/simulacao_chuvas_deslizamentos_rj.csv`: base **simulada** com 960 linhas (8 municípios × 120 meses, 2015–2024) e 14 colunas: ano, mês, data, município, região, população, chuva (mm), temperatura, ocorrências de deslizamento, desalojados, óbitos, nível de risco, índice de saturação do solo e umidade.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
sns.set_theme(style='whitegrid')

NOME = 'simulacao_chuvas_deslizamentos_rj.csv'
caminhos = ['../dados/' + NOME, 'dados/' + NOME, NOME]
caminho = next((c for c in caminhos if os.path.exists(c)), None)
if caminho is None:
    try:  # Google Colab: envie o CSV pela janela que vai abrir
        from google.colab import files
        files.upload()
        caminho = NOME
    except ImportError:
        raise FileNotFoundError('Coloque o CSV em dados/ ou na mesma pasta do notebook.')
df = pd.read_csv(caminho, parse_dates=['data'])
df.head()

## 3. Leitura e inspeção

In [ ]:
print(df.shape); print(df.dtypes); df.describe().T

## 4. Limpeza e preparação

In [ ]:
print('Nulos:', df.isna().sum().sum(), '| Duplicados:', df.duplicated().sum())
ordem = ['Baixo','Médio','Alto','Crítico']
df['nivel_risco'] = pd.Categorical(df['nivel_risco'], ordem, ordered=True)
# outliers de chuva (IQR)
q1,q3 = df.chuva_mm.quantile([.25,.75]); print('Outliers de chuva:', (df.chuva_mm > q3+1.5*(q3-q1)).sum())
# população varia mês a mês dentro do mesmo município (artefato da simulação): não será usada em taxas
print(df.groupby('municipio').populacao.nunique().min(), 'valores distintos de população por município')

## 5. Engenharia de atributos

In [ ]:
MESES=['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
df['mes_nome'] = df.mes.map(lambda m: MESES[m-1])
df['estacao'] = np.where(df.mes.isin([12,1,2,3]), 'Chuvosa (dez-mar)', 'Seca (abr-nov)')
df['desl_por_100mm'] = df.ocorrencias_deslizamento / df.chuva_mm.replace(0, np.nan) * 100
df[['data','municipio','estacao','desl_por_100mm']].head()

## 6. KPIs

In [ ]:
rank = df.groupby('municipio').ocorrencias_deslizamento.sum().sort_values(ascending=False)
kpis = {'Volume total de chuva (mm)': df.chuva_mm.sum(), 'Média de chuva (mm/mês/município)': df.chuva_mm.mean(),
 'Total de deslizamentos': df.ocorrencias_deslizamento.sum(), 'Município mais crítico': rank.index[0],
 'Total de desalojados': df.desalojados.sum(), 'Correlação chuva x deslizamentos': df.chuva_mm.corr(df.ocorrencias_deslizamento)}
pd.Series(kpis)

## 7. Análise exploratória e gráficos
### 7.1 Evolução temporal

In [ ]:
m = df.groupby('data').agg(chuva=('chuva_mm','mean'), desl=('ocorrencias_deslizamento','sum'))
fig, ax = plt.subplots(1,2, figsize=(13,3.5))
m.chuva.plot(ax=ax[0], color='#2b6cb0', title='Chuva média mensal (mm)')
m.desl.plot(ax=ax[1], color='#c53030', title='Deslizamentos por mês')
plt.tight_layout(); plt.show()

### 7.2 Comparação entre municípios

In [ ]:
mun = df.groupby(['municipio','regiao_rj']).agg(chuva=('chuva_mm','sum'), desl=('ocorrencias_deslizamento','sum'), desalojados=('desalojados','sum'), obitos=('obitos','sum')).reset_index()
mun['desl_por_100mm'] = (mun.desl/mun.chuva*100).round(2)
mun = mun.sort_values('desl', ascending=False)
fig, ax = plt.subplots(1,2, figsize=(13,4))
sns.barplot(data=mun, x='chuva', y='municipio', hue='regiao_rj', dodge=False, ax=ax[0]); ax[0].set_title('Chuva acumulada')
sns.barplot(data=mun, x='desl', y='municipio', hue='regiao_rj', dodge=False, ax=ax[1]); ax[1].set_title('Deslizamentos acumulados')
plt.tight_layout(); plt.show(); mun

### 7.3 Correlação

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4.5))
sns.scatterplot(data=df, x='chuva_mm', y='ocorrencias_deslizamento', hue='regiao_rj', alpha=.6, ax=ax[0])
cols=['chuva_mm','temperatura_media','umidade','indice_solo','ocorrencias_deslizamento','desalojados','obitos']
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax[1])
plt.tight_layout(); plt.show()
print(df.groupby('regiao_rj').apply(lambda g: g.chuva_mm.corr(g.ocorrencias_deslizamento)).round(2))

### 7.4 Sazonalidade e nível de risco

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,4.5))
p = df.pivot_table(index='ano', columns='mes', values='ocorrencias_deslizamento', aggfunc='sum')
sns.heatmap(p, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax[0]); ax[0].set_title('Deslizamentos ano x mês')
sns.boxplot(data=df, x='nivel_risco', y='chuva_mm', ax=ax[1]); ax[1].set_title('Chuva por nível de risco')
plt.tight_layout(); plt.show()
df.groupby('estacao')[['chuva_mm','ocorrencias_deslizamento']].mean().round(1)

### 7.5 Municípios fora do padrão

In [ ]:
# resíduo do modelo linear global chuva -> deslizamentos, médio por município
b, a = np.polyfit(df.chuva_mm, df.ocorrencias_deslizamento, 1)
df['residuo'] = df.ocorrencias_deslizamento - (a + b*df.chuva_mm)
df.groupby('municipio').residuo.mean().sort_values(ascending=False).round(2)

## 8. Interpretação dos resultados
- **Chuva:** os municípios têm volumes acumulados parecidos (≈14 a 20 mil mm), mas os deslizamentos **não** acompanham o volume: a região Serrana (Teresópolis, Petrópolis, Nova Friburgo) tem mais que o dobro dos deslizamentos dos demais.
- **Sazonalidade:** dezembro a março concentra chuva média ≈ 214 mm/mês e ≈ 10 deslizamentos/mês, contra ≈ 98 mm e ≈ 3,5 no restante do ano.
- **Correlação:** chuva × deslizamentos r ≈ 0,82; o índice de saturação do solo (r ≈ 0,89 com a chuva) reforça o mecanismo físico.
- **Fora do padrão:** os resíduos positivos da Serra mostram que, para a mesma chuva, ali ocorrem mais deslizamentos (relevo e ocupação de encostas).
- **Risco:** a chuva média cresce de Baixo (≈ 40 mm) a Crítico (≈ 203 mm).

## 9. Conclusão
O risco de deslizamentos no RJ combina **chuva intensa de verão** com **vulnerabilidade do terreno serrano**. Recomenda-se priorizar alertas e contenção na Serra antes de dezembro. Limitação: base simulada (população inconsistente entre meses, sem coordenadas); resultados são didáticos e devem ser confirmados com dados oficiais (INMET, CEMADEN, Defesa Civil).